In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:

def calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names, subject_index=2, take_abs=False):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:


            perturbed_data = np.load(f"perturbed_predictions/perturbed_prediction_dict_{band_name}_channel_wise_phase_shift_{factor}°_subject_{subject_index}.npy", allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = perturbed_data[ch_name][0]
                if take_abs:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else:
                    diff = pred_label_original - perturbed_amplitude
              
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

In [ ]:

def get_prediction_diff(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors):
    prediction_diff_mean = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        prediction_diff_mean[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in amplification_factors:
            mean_diffs = mean_diff_per_channel[band_name][factor]
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Store the prediction differences without sorting
            prediction_diff_mean[band_name][factor] = mean_diffs
            prediction_diff_median[band_name][factor] = median_diffs
    
    return prediction_diff_mean, prediction_diff_median

#prediction_diff_mean_unsorted, prediction_diff_median_unsorted = get_prediction_diff(mean_diff_per_channel, median_diff_per_channel, freq_bands, #phase_peturbations)

In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2):
    data_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/explanation_data"
    file_path = os.path.join(data_dir, f"subject_{subject_index}_results.pkl")

    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f)   
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
freq_bands = {
              "theta": (0, 4),
              "delta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
phase_peturbations = np.arange(45, 316, 45)

In [ ]:
cfg = load_config()
subject_phase_shift_results_median = {}
subject_phase_shift_results_mean = {}
for subject_index in cfg.dataset.test_subject_indices:
    original_predictions, _, explanations, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_predictions, freq_bands, phase_peturbations, ch_names, subject_index=subject_index)
    prediction_diff_mean_unsorted, prediction_diff_median_unsorted = get_prediction_diff(mean_diff_per_channel, median_diff_per_channel, freq_bands, phase_peturbations)
    #top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=10)
    subject_phase_shift_results_median[subject_index] = prediction_diff_median_unsorted
    subject_phase_shift_results_mean[subject_index] = prediction_diff_mean_unsorted


In [ ]:
from scipy.stats import spearmanr

def compute_phase_shift_correlations(results_dict, correlation_type='pearson', subjects=None, bands=None):
    """
    Compute correlations between phase shifts for each subject and frequency band.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary containing results for all subjects, bands, and phase shifts
    correlation_type : str
        'pearson' or 'spearman' correlation
    subjects : list or None
        List of subject IDs to analyze. If None, analyze all subjects.
    bands : list or None
        List of frequency bands to analyze. If None, analyze all bands.
        
    Returns:
    --------
    dict
        Nested dictionary with correlation matrices for each subject and band
    """
    
    if subjects is None:
        subjects = list(results_dict.keys())
    
    # Determine frequency bands from the first subject if not specified
    if bands is None and subjects:
        bands = list(results_dict[subjects[0]].keys())
    
    correlations = {}
    
    for subject_id in subjects:
        correlations[subject_id] = {}
        
        for band in bands:
            # Get the phase shifts for this subject and band
            phase_shifts = list(results_dict[subject_id][band].keys())
            n_phases = len(phase_shifts)
            
            # Initialize correlation matrix
            corr_matrix = np.zeros((n_phases, n_phases))
            
            # Compute correlations for each pair of phase shifts
            for i, phase1 in enumerate(phase_shifts):
                for j, phase2 in enumerate(phase_shifts):
                    if i == j:
                        # Perfect correlation with itself
                        corr_matrix[i, j] = 1.0
                    else:
                        # Extract channel values for both phase shifts
                        values1 = np.array(list(results_dict[subject_id][band][phase1].values()))
                        values2 = np.array(list(results_dict[subject_id][band][phase2].values()))
                        
                        # Calculate correlation based on specified type
                        if correlation_type == 'pearson':
                            corr = np.corrcoef(values1, values2)[0, 1]
                        elif correlation_type == 'spearman':
                            corr = spearmanr(values1, values2)[0]
                        else:
                            raise ValueError(f"Unknown correlation type: {correlation_type}")
                        
                        corr_matrix[i, j] = corr
            
            correlations[subject_id][band] = {
                'matrix': corr_matrix,

            }
    
    return correlations

In [ ]:
correlations = compute_phase_shift_correlations(subject_phase_shift_results_median, correlation_type='spearman', subjects=None, bands=None)

In [ ]:
def plot_correlation_matrix(correlation_matrix, phase_shifts, band_name, subject_id, save_path=None):
    """
    Plot a correlation matrix.
    
    Parameters:
    -----------
    correlation_matrix : np.ndarray
        Matrix of correlation coefficients
    phase_shifts : list
        List of phase shift values
    band_name : str
        Name of the frequency band
    subject_id : int
        Subject identifier
    save_path : str, optional
        Path to save the figure
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    
    # Set axis labels
    ax.set_xticks(np.arange(len(phase_shifts)))
    ax.set_yticks(np.arange(len(phase_shifts)))
    ax.set_xticklabels(phase_shifts)
    ax.set_yticklabels(phase_shifts)
    ax.set_xlabel('Phase shift (degrees)')
    ax.set_ylabel('Phase shift (degrees)')
    ax.set_title(f'{band_name.capitalize()} band: Spearman correlation')
    
    # Add colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Correlation coefficient')
    
    # Rotate the tick labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Loop over data dimensions and create text annotations
    for i in range(len(phase_shifts)):
        for j in range(len(phase_shifts)):
            text = ax.text(j, i, f"{correlation_matrix[i, j]:.2f}",
                          ha="center", va="center", color="black" if abs(correlation_matrix[i, j]) < 0.7 else "white")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    return fig, ax

def plot_all_freq_bands_correlation(correlations, subject_id, phase_shifts, save_path=None):
    """
    Plot correlation matrices for all frequency bands of a given subject.
    
    Parameters:
    -----------
    correlations : dict
        Dictionary containing correlation matrices for different subjects and bands
    subject_id : int
        Subject identifier
    phase_shifts : array
        Array of phase shift values
    save_path : str, optional
        Path to save the figure
    """
    bands = list(correlations[subject_id].keys())
    
    fig, axs = plt.subplots(1, len(bands), figsize=(20, 4))
    
    for i, band in enumerate(bands):
        ax = axs[i]
        corr_matrix = correlations[subject_id][band]['matrix']
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        
        # Set axis labels
        ax.set_xticks(np.arange(len(phase_shifts)))
        ax.set_yticks(np.arange(len(phase_shifts)))
        ax.set_xticklabels(phase_shifts)
        ax.set_yticklabels(phase_shifts)
        
        if i == 0:
            ax.set_ylabel('Phase shift (degrees)')
        
        ax.set_xlabel('Phase shift (degrees)')
        ax.set_title(f'{band.capitalize()}')
        
        # Rotate the tick labels
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Add colorbar on the right side
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='Correlation coefficient')
    
    plt.suptitle(f'Subject {subject_id}: Phase Shift Correlation Across Frequency Bands', fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    
    if save_path:
        plt.savefig(save_path)
    
    return fig, axs

In [ ]:
def ag
    return fig, axs
    
        plt.savefig(save_path)
    if save_path:
    
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    plt.suptitle('Mean Phase Shift Correlations Across All Subjects', fontsize=16)
    
    cbar = fig.colorbar(im, cax=cbar_ax, label='Mean correlation coefficient')
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    # Add colorbar
    
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        # Rotate tick labels
        
                             ha="center", va="center", color="black" if abs(mean_matrix[i, j]) < 0.7 else "white")
                text = ax.text(j, i, f"{mean_matrix[i, j]:.2f}",
            for j in range(len(phase_shifts)):
        for i in range(len(phase_shifts)):
        # Add text annotations
        
        ax.set_title(f'{band.capitalize()}')
        ax.set_xlabel('Phase shift (degrees)')
        
            ax.set_ylabel('Phase shift (degrees)')
        if b == 0:
        
        ax.set_yticklabels(phase_shifts)
        ax.set_xticklabels(phase_shifts)
        ax.set_yticks(np.arange(len(phase_shifts)))
        ax.set_xticks(np.arange(len(phase_shifts)))
        # Set axis labels
        
        im = ax.imshow(mean_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        # Plot the mean correlation matrix
        
        mean_matrix = mean_corr_matrices[b]
        ax = axs[b]
    for b, band in enumerate(bands):
    
    fig, axs = plt.subplots(1, n_bands, figsize=fig_size)
    # Plot the mean correlation matrices
    
            mean_corr_matrices[b] = np.mean(all_subject_matrices, axis=0)
        if len(all_subject_matrices) > 0:
        # Calculate mean across subjects
        
        all_subject_matrices = np.array(all_subject_matrices)
        # Convert to array for calculations
        
                all_subject_matrices.append(matrix)
                matrix = correlations[subject_id][band]['matrix']
            if band in correlations[subject_id]:
            # Check if this subject has data for this band
        for subject_id in subjects:
        
        all_subject_matrices = []
        # Collect matrices for all subjects for this band
    for b, band in enumerate(bands):
    # Aggregate correlations across subjects
    
    mean_corr_matrices = np.zeros((n_bands, n_phases, n_phases))
    # Create array for mean
    
    n_bands = len(bands)
    n_phases = len(phase_shifts)
    # Initialize arrays to store aggregated results
    
        bands = list(correlations[subjects[0]].keys())
    if bands is None and subjects:
    # Determine frequency bands if not specified
    
    subjects = list(correlations.keys())
    
    """
        Figure and axes objects
    tuple
    --------
    Returns:
        
        Path to save the figure
    save_path : str, optional
        Figure size for the plot
    fig_size : tuple
        List of frequency bands to analyze. If None, analyze all bands.
    bands : list or None
        Array of phase shift values
    
    plt.suptitle('Mean Phase Shift Correlations Across All Subjects', fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    
    if save_path:
        plt.savefig(save_path)
    
    return fig, axs
    phase_shifts : array
        Dictionary containing correlation matrices for different subjects and bands
    correlations : dict
    -----------
    Parameters:
    
    Aggregate correlation matrices across all subjects and plot mean correlations.
    """gregate_and_plot_correlations(correlations, phase_shifts, bands=None, fig_size=(20, 6), save_path=None):c'Mean correlation coefficient')

In [ ]:
plot_correlation_matrix(correlations[2]['theta']['matrix'], phase_peturbations, 'theta', 2)

In [ ]:
plot_all_freq_bands_correlation(correlations, 2, phase_peturbations)

In [ ]:
for subject_id in correlations.keys():
    plot_all_freq_bands_correlation(correlations, subject_id, phase_peturbations)

there does not seem to be much of an expected cyclical effect.
Howver for far distances of phase shifts (mostly 45° vs 315°) there is usually an anticorrelation

In [ ]:
correlations = compute_phase_shift_correlations(subject_phase_shift_results_median, correlation_type='pearson', subjects=None, bands=None)

In [ ]:
for subject_id in correlations.keys():
    plot_all_freq_bands_correlation(correlations, subject_id, phase_peturbations)